In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/pre_cell_6.pickle

In [ ]:
%%PandasProfile
### cell 6 ###

# Compute los_diff in one vectorized step, applying -1 multiplier for 'left' directions in the same pass
# (avoids two separate assignments and avoids method-call overhead from sub/mul)
sign = 1 - 2 * (df['playDirection'] == 'left')
df['los_diff'] = (df['x'] - df['los']) * sign

# Merge possessionTeam onto df via a join on the two key columns (inner join by default)
# using copy=False to avoid unnecessary data copying
df = df.merge(
    plays[['gameId', 'playId', 'possessionTeam']],
    on=['gameId', 'playId'],
    how='inner',
    copy=False
)